# Sid — Historical Feature Engineering & Preprocessing

Builds a rich feature set combining:
- MIT Election Lab historical voting patterns (2000–2020)
- Engineered political trend features (swing, momentum, volatility)
- ACS 2020 demographic and socioeconomic features

**Target:** Predict 2024 county-level presidential election outcomes
**Output:** `../data/processed/historical_merged_dataset.csv`

In [1]:
import pandas as pd
import numpy as np
import os
from functools import reduce

### (2) File paths

In [2]:
MIT_PATH        = "../data/Raw/countypres_2000-2024.csv"

AGE_PATH        = "../data/Raw/demographic/ACSDT5Y2020.B01001-Data.csv"
RACE_PATH       = "../data/Raw/demographic/ACSDT5Y2020.B02001-Data.csv"
EDUCATION_PATH  = "../data/Raw/demographic/ACSDT5Y2020.B15003-Data.csv"
INCOME_PATH     = "../data/Raw/socioeconomic/ACSDT5Y2020.B19013-Data.csv"
POVERTY_PATH    = "../data/Raw/socioeconomic/ACSDT5Y2020.B17001-Data.csv"
EMPLOYMENT_PATH = "../data/Raw/socioeconomic/ACSDT5Y2020.B23025-Data.csv"
HOUSING_PATH    = "../data/Raw/socioeconomic/ACSDT5Y2020.B25001-Data.csv"

OUTPUT_PATH     = "../data/processed/historical_merged_dataset.csv"

TRAIN_YEARS = [2000, 2004, 2008, 2012, 2016, 2020]
TEST_YEAR   = 2024

### (3) Load raw data

In [3]:
mit_df        = pd.read_csv(MIT_PATH)
age_df        = pd.read_csv(AGE_PATH)
race_df       = pd.read_csv(RACE_PATH)
education_df  = pd.read_csv(EDUCATION_PATH)
income_df     = pd.read_csv(INCOME_PATH)
poverty_df    = pd.read_csv(POVERTY_PATH)
employment_df = pd.read_csv(EMPLOYMENT_PATH)
housing_df    = pd.read_csv(HOUSING_PATH)

print("Loaded shapes:")
for name, df in [("MIT", mit_df), ("Age", age_df), ("Race", race_df),
                  ("Education", education_df), ("Income", income_df),
                  ("Poverty", poverty_df), ("Employment", employment_df),
                  ("Housing", housing_df)]:
    print(f"  {name}: {df.shape}")

Loaded shapes:
  MIT: (94151, 12)
  Age: (3222, 101)
  Race: (3222, 23)
  Education: (3222, 53)
  Income: (3222, 5)
  Poverty: (3222, 121)
  Employment: (3222, 17)
  Housing: (3222, 5)


### (4) Clean MIT FIPS

In [4]:
mit_df = mit_df[mit_df["county_fips"].notna()].copy()
mit_df["county_fips"] = (
    mit_df["county_fips"]
    .astype(float).astype(int).astype(str).str.zfill(5)
)

print(f"MIT years available: {sorted(mit_df['year'].unique())}")
print(f"MIT total rows: {len(mit_df)}")

MIT years available: [np.int64(2000), np.int64(2004), np.int64(2008), np.int64(2012), np.int64(2016), np.int64(2020), np.int64(2024)]
MIT total rows: 94099


### (5) Helper — get one election year pivot

In [5]:
def get_election_year(df, year):
    mit_year = df[df["year"] == year].copy()
    mit_year = mit_year[mit_year["party"].isin(["DEMOCRAT", "REPUBLICAN"])]

    # Use TOTAL mode where available, otherwise sum all modes
    has_total = set(mit_year[mit_year["mode"] == "TOTAL"]["county_fips"])
    mit_total     = mit_year[mit_year["mode"] == "TOTAL"]
    mit_non_total = mit_year[~mit_year["county_fips"].isin(has_total)]
    mit_year = pd.concat([mit_total, mit_non_total], ignore_index=True)

    pivot = mit_year.pivot_table(
        index=["county_fips", "state", "county_name", "totalvotes"],
        columns="party",
        values="candidatevotes",
        aggfunc="sum"
    ).reset_index()

    pivot.columns.name = None
    pivot = pivot.rename(columns={
        "DEMOCRAT":   "democrat_votes",
        "REPUBLICAN": "republican_votes"
    })
    pivot["democrat_votes"]   = pivot["democrat_votes"].fillna(0)
    pivot["republican_votes"] = pivot["republican_votes"].fillna(0)
    pivot["dem_share"]        = pivot["democrat_votes"] / pivot["totalvotes"]
    pivot["winner"]           = np.where(
        pivot["democrat_votes"] > pivot["republican_votes"], 1, 0
    )
    return pivot

### (6) Build per-year historical features (2000–2020)

In [6]:
yearly_features = []

for year in TRAIN_YEARS:
    pivot = get_election_year(mit_df, year)
    keep  = ["county_fips",
             f"dem_share_{year}",
             f"winner_{year}",
             f"totalvotes_{year}"]
    pivot = pivot.rename(columns={
        "dem_share":   f"dem_share_{year}",
        "winner":      f"winner_{year}",
        "totalvotes":  f"totalvotes_{year}"
    })
    yearly_features.append(pivot[keep])
    print(f"{year}: {len(pivot)} counties")

# Merge all years on county_fips
hist_df = reduce(
    lambda left, right: pd.merge(left, right, on="county_fips", how="inner"),
    yearly_features
)

print(f"\nHistorical base shape: {hist_df.shape}")

2000: 3154 counties
2004: 3155 counties
2008: 3155 counties
2012: 3155 counties
2016: 3155 counties
2020: 3154 counties

Historical base shape: (3152, 19)


### (7) Engineer advanced political trend features

In [7]:
dem_share_cols = [f"dem_share_{y}" for y in TRAIN_YEARS]
winner_cols    = [f"winner_{y}"    for y in TRAIN_YEARS]

# --- Basic aggregates ---
hist_df["dem_wins_count"]  = hist_df[winner_cols].sum(axis=1)
hist_df["avg_dem_share"]   = hist_df[dem_share_cols].mean(axis=1)

# --- Trend: overall direction 2000 → 2020 ---
hist_df["dem_share_trend"] = (
    hist_df["dem_share_2020"] - hist_df["dem_share_2000"]
)

# --- Momentum: most recent shift 2016 → 2020 ---
hist_df["momentum_2016_2020"] = (
    hist_df["dem_share_2020"] - hist_df["dem_share_2016"]
)

# --- Momentum: shift 2012 → 2016 ---
hist_df["momentum_2012_2016"] = (
    hist_df["dem_share_2016"] - hist_df["dem_share_2012"]
)

# --- Volatility: how much vote share fluctuates ---
hist_df["vote_volatility"] = hist_df[dem_share_cols].std(axis=1)

# --- Competitiveness: how many elections were within 5% ---
def count_competitive(row):
    return sum(abs(row[col] - 0.5) <= 0.05 for col in dem_share_cols)

hist_df["competitive_count"] = hist_df.apply(count_competitive, axis=1)

# --- Political alignment: consistent Dem, consistent Rep, or swing ---
def alignment(row):
    wins = row["dem_wins_count"]
    if wins == 6:   return "Strong Democrat"
    elif wins >= 4: return "Lean Democrat"
    elif wins == 3: return "Swing"
    elif wins >= 1: return "Lean Republican"
    else:           return "Strong Republican"

hist_df["political_alignment"] = hist_df.apply(alignment, axis=1)

print("Advanced features engineered:")
new_cols = ["dem_wins_count", "avg_dem_share", "dem_share_trend",
            "momentum_2016_2020", "momentum_2012_2016",
            "vote_volatility", "competitive_count", "political_alignment"]
print(hist_df[new_cols].describe().round(3))
print(f"\nAlignment distribution:")
print(hist_df["political_alignment"].value_counts())

Advanced features engineered:
       dem_wins_count  avg_dem_share  dem_share_trend  momentum_2016_2020  \
count        3152.000       3152.000         3150.000            3152.000   
mean            1.239          0.372           -0.063               0.018   
std             2.078          0.133            0.117               0.030   
min             0.000          0.064           -0.482              -0.271   
25%             0.000          0.280           -0.137              -0.000   
50%             0.000          0.360           -0.070               0.016   
75%             2.000          0.448            0.008               0.035   
max             6.000          0.901            0.485               0.130   

       momentum_2012_2016  vote_volatility  competitive_count  
count            3151.000         3152.000           3152.000  
mean               -0.069            0.058              0.994  
std                 0.050            0.029              1.493  
min                -

### (8) Clean ACS tables

In [8]:
ACS_SENTINEL = -666666666

def clean_acs(df, label):
    df = df.iloc[1:].copy()
    df["county_fips"] = df["GEO_ID"].astype(str).str[-5:]
    skip = {"GEO_ID", "NAME", "county_fips"}
    est_cols = list(dict.fromkeys([
        c for c in df.columns
        if str(c).endswith("E") and c not in skip
    ]))
    df = df[["GEO_ID", "NAME", "county_fips"] + est_cols].copy()
    for col in est_cols:
        if isinstance(df[col], pd.Series):
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df[est_cols] = df[est_cols].replace(ACS_SENTINEL, np.nan)
    df["county_fips"] = df["county_fips"].astype(str).str.zfill(5)
    print(f"  {label}: {df.shape}")
    return df

print("Cleaning ACS tables:")
age_clean        = clean_acs(age_df,        "Age")
race_clean       = clean_acs(race_df,       "Race")
education_clean  = clean_acs(education_df,  "Education")
income_clean     = clean_acs(income_df,     "Income")
poverty_clean    = clean_acs(poverty_df,    "Poverty")
employment_clean = clean_acs(employment_df, "Employment")
housing_clean    = clean_acs(housing_df,    "Housing")

Cleaning ACS tables:
  Age: (3221, 52)
  Race: (3221, 13)
  Education: (3221, 28)
  Income: (3221, 4)
  Poverty: (3221, 62)
  Employment: (3221, 10)
  Housing: (3221, 4)


### (9) Merge ACS tables and engineer demographic features

In [9]:
def slim(df):
    return df.drop(columns=["GEO_ID", "NAME"], errors="ignore")

acs_merged = age_clean.copy()
for df, label in [
    (race_clean,       "Race"),
    (education_clean,  "Education"),
    (income_clean,     "Income"),
    (poverty_clean,    "Poverty"),
    (employment_clean, "Employment"),
    (housing_clean,    "Housing"),
]:
    acs_merged = acs_merged.merge(slim(df), on="county_fips", how="inner")

print(f"ACS merged shape: {acs_merged.shape}")

# Engineer demographic rate features
acs_merged["total_population"]  = acs_merged["B01001_001E"]
acs_merged["median_income"]     = acs_merged["B19013_001E"]
acs_merged["total_housing"]     = acs_merged["B25001_001E"]
acs_merged["labor_force"]       = acs_merged["B23025_002E"]
acs_merged["unemployed"]        = acs_merged["B23025_005E"]
acs_merged["below_poverty"]     = acs_merged["B17001_002E"]
acs_merged["white_pop"]         = acs_merged["B02001_002E"]
acs_merged["black_pop"]         = acs_merged["B02001_003E"]
acs_merged["asian_pop"]         = acs_merged["B02001_005E"]
acs_merged["bachelors"]         = acs_merged["B15003_022E"]
acs_merged["masters"]           = acs_merged["B15003_023E"]
acs_merged["professional_deg"]  = acs_merged["B15003_024E"]
acs_merged["doctorate"]         = acs_merged["B15003_025E"]

pop = acs_merged["total_population"].replace(0, np.nan)
lf  = acs_merged["labor_force"].replace(0, np.nan)

acs_merged["higher_ed_rate"]    = (acs_merged["bachelors"] + acs_merged["masters"] +
                                    acs_merged["professional_deg"] + acs_merged["doctorate"]) / pop
acs_merged["poverty_rate"]      = acs_merged["below_poverty"] / pop
acs_merged["unemployment_rate"] = acs_merged["unemployed"] / lf
acs_merged["white_pct"]         = acs_merged["white_pop"] / pop
acs_merged["black_pct"]         = acs_merged["black_pop"] / pop
acs_merged["asian_pct"]         = acs_merged["asian_pop"] / pop
acs_merged["log_population"]    = np.log1p(acs_merged["total_population"])
acs_merged["housing_density"]   = acs_merged["total_housing"] / pop

demo_features = ["county_fips", "higher_ed_rate", "poverty_rate",
                 "unemployment_rate", "white_pct", "black_pct",
                 "asian_pct", "median_income", "log_population",
                 "housing_density"]

acs_slim = acs_merged[demo_features].copy()
print(f"Demographic features shape: {acs_slim.shape}")

ACS merged shape: (3221, 155)
Demographic features shape: (3221, 10)


### (10) Add interaction features

In [10]:
# Merge historical + demographic first
combined = hist_df.merge(acs_slim, on="county_fips", how="inner")
print(f"Combined shape before interactions: {combined.shape}")

# Interaction features — things Atif didn't build
# Education × trend: educated counties moving Democrat
combined["edu_x_trend"]       = combined["higher_ed_rate"] * combined["dem_share_trend"]

# White% × momentum: white counties that swung in 2020
combined["white_x_momentum"]  = combined["white_pct"] * combined["momentum_2016_2020"]

# Income × volatility: wealthy but unpredictable counties
combined["income_x_volatility"] = combined["median_income"] * combined["vote_volatility"]

# Education × wins: educated AND historically Democrat
combined["edu_x_wins"]        = combined["higher_ed_rate"] * combined["dem_wins_count"]

print("Interaction features added.")
print(f"Combined shape after interactions: {combined.shape}")

Combined shape before interactions: (3113, 36)
Interaction features added.
Combined shape after interactions: (3113, 40)


### (11) Add 2024 election as test target

In [11]:
pivot_2024 = get_election_year(mit_df, 2024)
pivot_2024 = pivot_2024.rename(columns={
    "dem_share": "dem_share_2024",
    "winner":    "party_winner_2024"
})

pivot_2024 = pivot_2024[[
    "county_fips", "state", "county_name",
    "totalvotes", "dem_share_2024", "party_winner_2024"
]]

print(f"2024 counties: {len(pivot_2024)}")
print(f"2024 winner distribution:")
print(pivot_2024["party_winner_2024"].value_counts().rename({1:"Democrat", 0:"Republican"}))

2024 counties: 3154
2024 winner distribution:
party_winner_2024
Republican    2685
Democrat       469
Name: count, dtype: int64


### (12) Final merge — all features + 2024 target

In [12]:
final_df = combined.merge(pivot_2024, on="county_fips", how="inner")

# Remove any duplicate columns
final_df = final_df.loc[:, ~final_df.columns.duplicated()]
final_df["county_fips"] = final_df["county_fips"].astype(str).str.zfill(5)

print(f"Final dataset shape: {final_df.shape}")
print(f"\nMissing values:")
missing = final_df.isnull().sum()
missing = missing[missing > 0]
print(missing if len(missing) > 0 else "None")

print(f"\n2024 Target distribution:")
print(final_df["party_winner_2024"].value_counts().rename({1:"Democrat", 0:"Republican"}))

Final dataset shape: (3113, 45)

Missing values:
median_income          1
income_x_volatility    1
dtype: int64

2024 Target distribution:
party_winner_2024
Republican    2661
Democrat       452
Name: count, dtype: int64


### (13) Save dataset

In [14]:
os.makedirs("../data/processed", exist_ok=True)
final_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Shape: {final_df.shape}")
print(f"\nFeature summary:")
print(f"  Historical features (per year): {len(TRAIN_YEARS) * 3}")
print(f"  Political trend features:       8")
print(f"  Demographic features:           9")
print(f"  Interaction features:           4")
print(f"  Target (2024):                  party_winner_2024, dem_share_2024")

Saved: ../data/processed/historical_merged_dataset.csv
Shape: (3113, 45)

Feature summary:
  Historical features (per year): 18
  Political trend features:       8
  Demographic features:           9
  Interaction features:           4
  Target (2024):                  party_winner_2024, dem_share_2024
